In [20]:
# NumPy is used for random-number generation and numerical calculations.
import numpy as np
# Pandas is used to create and manipulate the applicant dataset.
import pandas as pd
# os is used for creating folders and constructing file paths.
import os

# Fix the random seed so that the same synthetic dataset can be reproduced.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [21]:
# Create a folder inside the local Google Colab runtime.
# Files stored under /content are available during the current Colab session.
MODEL_DIR = '/content/faid_optimizer'
os.makedirs(MODEL_DIR, exist_ok=True)

# Define the local path where the generated synthetic CSV will be saved.
DATA_PATH = os.path.join(MODEL_DIR, 'tn_synthetic_aid_dataset.csv')
print('Dataset will be saved to:', DATA_PATH)

Dataset will be saved to: /content/faid_optimizer/tn_synthetic_aid_dataset.csv


In [22]:
# List of Tamil Nadu districts represented in the synthetic applicant pool.
DISTRICTS = ["Chennai","Coimbatore","Madurai","Tiruchirappalli","Salem","Tirunelveli",
             "Erode","Vellore","Thanjavur","Dindigul","Namakkal","Tiruppur","Theni",
             "Cuddalore","Karur","Nagapattinam","Virudhunagar","Kanyakumari","Villupuram","Thoothukudi"]

# Define three synthetic college tiers and five admission categories.
COLLEGE_TIER = ["Tier-1 (Autonomous)", "Tier-2 (Affiliated)", "Tier-3 (Self-financing)"]
CATEGORY = ["OC", "BC", "MBC", "SC", "ST"]

# Approximate category sampling probabilities used only for synthetic-data generation.
CATEGORY_P = [0.12, 0.30, 0.28, 0.20, 0.10]


In [23]:
# Generate a synthetic applicant dataset with a reproducible random process.
def generate_tn_dataset(n=6000, seed=RANDOM_STATE):
    # Create an independent random-number generator for reproducible results.
    rng = np.random.default_rng(seed)

    # Randomly assign each applicant a district and admission category.
    district = rng.choice(DISTRICTS, n)
    category = rng.choice(CATEGORY, n, p=CATEGORY_P)

    # Generate an urban/rural indicator: 1 = urban, 0 = rural.
    urban = rng.choice([0, 1], n, p=[0.45, 0.55])

    # Set different baseline family-income levels for urban and rural applicants.
    base_income = np.where(urban == 1, 420000, 260000)

    # Apply category-based income multipliers for synthetic calibration.
    cat_adj = pd.Series(category).map({"OC":1.5,"BC":1.15,"MBC":1.0,"SC":0.75,"ST":0.65}).values

    # Generate realistic-looking family income values using a log-normal distribution.
    family_income = np.clip(rng.lognormal(mean=np.log(base_income*cat_adj), sigma=0.55), 60000, 4500000).round(-3)

    # Generate first-generation college status and parent graduation status.
    first_gen = rng.choice([0, 1], n, p=[0.55, 0.45])
    parent_grad = np.where(first_gen == 1, 0, rng.choice([0, 1], n, p=[0.35, 0.65]))

    # Generate academic-strength indicators.
    cutoff_12th = np.clip(rng.normal(78, 10, n), 45, 99.9).round(2)
    entrance_score = np.clip(cutoff_12th*1.6 + rng.normal(0, 15, n), 30, 200).round(1)

    # Assign a college tier and generate tuition according to the tier.
    tier = rng.choice(COLLEGE_TIER, n, p=[0.2, 0.45, 0.35])
    tuition = pd.Series(tier).map({"Tier-1 (Autonomous)":180000, "Tier-2 (Affiliated)":110000, "Tier-3 (Self-financing)":85000}).values
    tuition = (tuition + rng.normal(0, 8000, n)).round(-3).clip(50000)

    # Generate distance from the college and the number of competing offers.
    distance_km = np.clip(rng.exponential(60, n), 2, 600).round(1)
    competing_offers = rng.poisson(1.4, n)

    # Convert income and academic score into normalized need and merit indicators.
    need_index = np.clip(1 - (family_income / family_income.max()), 0.02, 0.98)
    merit_index = np.clip((entrance_score - 30) / 170, 0.02, 0.98)

    # Generate synthetic merit-based and need-based aid percentages.
    merit_aid_pct = np.clip(merit_index*0.6 + rng.normal(0, 0.08, n), 0, 0.7)
    need_aid_pct = np.clip(need_index*0.55 + rng.normal(0, 0.08, n), 0, 0.75)

    # Combine the two aid components into a total aid percentage.
    total_aid_pct = np.clip(merit_aid_pct*0.5 + need_aid_pct*0.5, 0, 0.85)

    # Calculate the monetary aid amount and the resulting net price.
    aid_amount = (tuition * total_aid_pct).round(-2)
    net_price = (tuition - aid_amount).round(-2)

    # Estimate price sensitivity and affordability for the synthetic enrollment model.
    price_sensitivity = np.clip(0.55 + need_index*0.6 - merit_index*0.25, 0.15, 1.4)
    affordability = 1 - np.clip(net_price / (family_income*0.4 + 1), 0, 1.5)

    # Build a synthetic enrollment logit from affordability, aid burden, merit,
    # competing offers, distance, parent education, and random variation.
    logit = (
        -0.6
        + 2.6*affordability
        - 1.1*price_sensitivity*(net_price/tuition)
        + 0.9*merit_index
        - 0.18*competing_offers
        - 0.15*(distance_km/600)
        + 0.35*parent_grad
        + rng.normal(0, 0.5, n)
    )

    # Convert the logit score to an enrollment probability using the sigmoid function.
    prob_enroll = 1 / (1 + np.exp(-logit))

    # Sample the final binary enrollment label from the calculated probability.
    enrolled = rng.binomial(1, prob_enroll)

    # Assemble all generated variables into a Pandas DataFrame.
    df = pd.DataFrame({
        "district": district, "category": category, "urban": urban,
        "family_income": family_income, "first_gen": first_gen, "parent_grad": parent_grad,
        "cutoff_12th": cutoff_12th, "entrance_score": entrance_score,
        "college_tier": tier, "tuition": tuition, "distance_km": distance_km,
        "competing_offers": competing_offers, "merit_aid_pct": merit_aid_pct.round(3),
        "need_aid_pct": need_aid_pct.round(3), "total_aid_pct": total_aid_pct.round(3),
        "aid_amount": aid_amount, "net_price": net_price, "enrolled": enrolled
    })

    # Return the completed synthetic applicant dataset.
    return df


In [24]:
# Generate 6,000 synthetic applicant records using the fixed random seed.
# Using the same seed makes the generated dataset reproducible.
df = generate_tn_dataset(n=6000, seed=RANDOM_STATE)

# Save the generated dataset as a CSV file in the local Colab runtime.
df.to_csv(DATA_PATH, index=False)

# Display basic information so we can verify that generation and saving worked.
print('Saved ->', DATA_PATH)
print('Shape:', df.shape)
print('Overall yield rate:', round(df['enrolled'].mean(), 3))
df.head()


Saved -> /content/faid_optimizer/tn_synthetic_aid_dataset.csv
Shape: (6000, 18)
Overall yield rate: 0.563


,district,category,urban,family_income,first_gen,parent_grad,cutoff_12th,entrance_score,college_tier,tuition,distance_km,competing_offers,merit_aid_pct,need_aid_pct,total_aid_pct,aid_amount,net_price,enrolled
0,Coimbatore,SC,1,691000.0,0,1,92.35,145.6,Tier-2 (Affiliated),110000.0,29.8,3,0.443,0.430,0.437,48000.0,62000.0,1
1,Nagapattinam,MBC,0,686000.0,1,0,77.63,130.7,Tier-3 (Self-financing),91000.0,15.8,0,0.389,0.324,0.357,32500.0,58500.0,1
2,Cuddalore,BC,1,579000.0,0,1,68.22,104.2,Tier-2 (Affiliated),108000.0,66.9,0,0.292,0.422,0.357,38500.0,69500.0,1
3,Thanjavur,SC,0,99000.0,0,0,80.98,129.6,Tier-3 (Self-financing),91000.0,31.2,1,0.504,0.457,0.481,43800.0,47200.0,0
4,Thanjavur,OC,0,1034000.0,1,0,77.28,126.9,Tier-2 (Affiliated),119000.0,19.4,2,0.395,0.389,0.392,46600.0,72400.0,0


In [25]:
# Display the null or missing values in the dataset
print(df.isnull().sum().sum(), 'missing values')
print()

# Display the datatype of each columns in the dataset
print(df.dtypes)
print()

# Dispaly the distribution of each category
print('Category distribution:')
print(df['category'].value_counts(normalize=True).round(3))
print()

# Display the yield rate by college tier wise
print('Yield rate by college tier:')
print(df.groupby('college_tier')['enrolled'].mean().round(3))

0 missing values

district             object
category             object
urban                 int64
family_income       float64
first_gen             int64
parent_grad           int64
cutoff_12th         float64
entrance_score      float64
college_tier         object
tuition             float64
distance_km         float64
competing_offers      int64
merit_aid_pct       float64
need_aid_pct        float64
total_aid_pct       float64
aid_amount          float64
net_price           float64
enrolled              int64
dtype: object

Category distribution:
category
BC     0.308
MBC    0.283
SC     0.195
OC     0.117
ST     0.097
Name: proportion, dtype: float64

Yield rate by college tier:
college_tier
Tier-1 (Autonomous)        0.437
Tier-2 (Affiliated)        0.566
Tier-3 (Self-financing)    0.629
Name: enrolled, dtype: float64
